In [2]:
import os
import PIL.Image
import torch
import numpy as np
from transformers import AutoModelForCausalLM
from janus.models import MultiModalityCausalLM, VLChatProcessor
import time
import re


# Specify the path to the model
model_path = "deepseek-ai/Janus-Pro-7B"
vl_chat_processor: VLChatProcessor = VLChatProcessor.from_pretrained(model_path)
tokenizer = vl_chat_processor.tokenizer

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message.


In [3]:
from datasets import load_dataset

dset = load_dataset("Jiwon-Kang/pixmo-point-count-concat_0-20", split='train', streaming=True)

Resolving data files:   0%|          | 0/559 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/559 [00:00<?, ?it/s]

In [4]:
from IPython.display import display
iter_dset = iter(dset)
for sample in iter_dset:
    count = sample['count']
    if count >= 18:
        first_sample = sample
        break



image = first_sample['image']
question = first_sample['question']
answer = first_sample['answer']
print(f"question: {question}")
print(f"answer: {answer}")
# display(image)

question: Locate all Video games and count the total number of Video games
answer: <points x1="78.5" y1="74.4" x2="80.5" y2="74.8" x3="82.8" y3="74.4" x4="86.2" y4="74.8" x5="89.0" y5="75.4" x6="91.8" y6="75.0" x7="95.2" y7="74.5" x8="98.0" y8="72.1" x9="99.8" y9="64.6" x10="83.0" y10="86.1" x11="82.8" y11="88.7" x12="82.8" y12="91.5" x13="82.9" y13="94.2" x14="42.2" y14="84.8" x15="41.0" y15="87.4" x16="40.5" y16="89.5" x17="40.6" y17="91.5" x18="40.2" y18="95.7" alt=Video games>Video games</points>. Total number of Video games is **18**.


In [8]:
from janus.utils.io import load_pil_images

conversation = [
    {
        "role": "User",
        "content": f"<image_placeholder>\n{question}",
        "images": ["images/equation.png"],
    },
    # {"role": "Assistant", "content": f"{answer}"},
]

# load images and prepare for inputs
# pil_images = load_pil_images(conversation)
pil_images = [image]

prepare_inputs = vl_chat_processor(
    conversations=conversation, images=pil_images, force_batchify=True
)

# Get input IDs and token count
input_ids = prepare_inputs.input_ids
token_count = input_ids.shape[1]

# print(f"Input tokens: {input_ids}")
print(f"Number of tokens: {token_count}")

    

sft_prompt: You are a helpful language and vision assistant. You are able to understand the visual content that the user provides, and assist the user with a variety of tasks using natural language.

User: <image_placeholder>
Locate all Video games and count the total number of Video games
Number of tokens: 633


In [10]:
from janus.utils.io import load_pil_images

conversation = [
    {
        "role": "<|User|>",
        "content": f"<image_placeholder>\n{question}",
        "images": ["images/equation.png"],
    },
    {"role": "<|Assistant|>", "content": f"{answer}"},
]

# load images and prepare for inputs
# pil_images = load_pil_images(conversation)
pil_images = [image]

prepare_inputs = vl_chat_processor(
    conversations=conversation, images=pil_images, force_batchify=True
)

# Get input IDs and token count
input_ids = prepare_inputs.input_ids
token_count = input_ids.shape[1]

# print(f"Input tokens: {input_ids}")
print(f"Number of tokens: {token_count}")

    

sft_prompt: You are a helpful language and vision assistant. You are able to understand the visual content that the user provides, and assist the user with a variety of tasks using natural language.

<|User|>: <image_placeholder>
Locate all Video games and count the total number of Video games

<|Assistant|>: <points x1="78.5" y1="74.4" x2="80.5" y2="74.8" x3="82.8" y3="74.4" x4="86.2" y4="74.8" x5="89.0" y5="75.4" x6="91.8" y6="75.0" x7="95.2" y7="74.5" x8="98.0" y8="72.1" x9="99.8" y9="64.6" x10="83.0" y10="86.1" x11="82.8" y11="88.7" x12="82.8" y12="91.5" x13="82.9" y13="94.2" x14="42.2" y14="84.8" x15="41.0" y15="87.4" x16="40.5" y16="89.5" x17="40.6" y17="91.5" x18="40.2" y18="95.7" alt=Video games>Video games</points>. Total number of Video games is **18**.<｜end▁of▁sentence｜>
Number of tokens: 966


In [ ]:
tokenizer = vl_chat_processor.tokenizer
print("=== Special Tokens Map ===")
print(tokenizer.special_tokens_map)

print("\n=== Key Special Tokens ===")
print(f"EOS Token: {tokenizer.eos_token} | ID: {tokenizer.eos_token_id}")
print(f"BOS Token: {tokenizer.bos_token} | ID: {tokenizer.bos_token_id}")
print(f"PAD Token: {tokenizer.pad_token} | ID: {tokenizer.pad_token_id}")
print(f"UNK Token: {tokenizer.unk_token} | ID: {tokenizer.unk_token_id}")

print("\n=== Additional Special Tokens ===")
print(f"Tokens: {tokenizer.additional_special_tokens}")
print(f"IDs:    {tokenizer.additional_special_tokens_ids}")

=== Special Tokens Map ===
{'bos_token': '<｜begin▁of▁sentence｜>', 'eos_token': '<｜end▁of▁sentence｜>', 'pad_token': '<｜▁pad▁｜>', 'additional_special_tokens': ['<image_placeholder>', '<patch_placeholder>', '<|ref|>', '<|/ref|>', '<|det|>', '<|/det|>', '<|grounding|>', '<|User|>', '<|Assistant|>']}

=== Key Special Tokens ===
EOS Token: <｜end▁of▁sentence｜> | ID: 100001
BOS Token: <｜begin▁of▁sentence｜> | ID: 100000
PAD Token: <｜▁pad▁｜> | ID: 100015
UNK Token: None | ID: None

=== Additional Special Tokens ===
Tokens: ['<image_placeholder>', '<patch_placeholder>', '<|ref|>', '<|/ref|>', '<|det|>', '<|/det|>', '<|grounding|>', '<|User|>', '<|Assistant|>']
IDs:    [100594, 100595, 100596, 100597, 100598, 100599, 100600, 100601, 100602]
